# GAT with SAGPool Session Readout

This Kaggle-ready notebook prepares raw Yoochoose and Diginetica inputs and prepares train/test prefix-label examples in one file. The SAGPool model definition and training loop will be added later.

Expected Kaggle input directories:
- `/kaggle/input/datasets/chadgostopp/recsys-challenge-2015` containing `yoochoose-clicks.dat`
- `/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset` containing `train-item-views.csv`

## Prepare Datasets

The Kaggle inputs contain the original raw files. This section reads the raw Kaggle files directly and prepares the in-memory prefix-label examples used by the following sections.

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")

YOOCHOOSE_INPUT_DIR = KAGGLE_INPUT_DIR / "datasets/chadgostopp/recsys-challenge-2015"
DIGINETICA_INPUT_DIR = (
    KAGGLE_INPUT_DIR / "datasets/profalbusdumbledore/diginetica-dataset"
)
YOOCHOOSE_SOURCE = YOOCHOOSE_INPUT_DIR / "yoochoose-clicks.dat"
DIGINETICA_SOURCE = DIGINETICA_INPUT_DIR / "train-item-views.csv"

print(f"Using Yoochoose source: {YOOCHOOSE_SOURCE}")
print(f"Using Diginetica source: {DIGINETICA_SOURCE}")

## Preprocess Sessions

The preprocessing flow builds ordered sessions, removes short sessions and rare items, splits chronologically, remaps item ids from training data, and expands sessions into prefix-label examples.

In [ ]:
def load_yoochoose_sessions(path):
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["session_id", "timestamp", "item_id"],
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)

    session_items = {}
    session_dates = {}
    current_session_id = None
    current_timestamp = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_timestamp is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_timestamp

        current_session_id = session_id
        current_timestamp = row.timestamp

        if session_id in session_items:
            session_items[session_id].append(row.item_id)
        else:
            session_items[session_id] = [row.item_id]

    if current_session_id is not None:
        session_dates[current_session_id] = current_timestamp

    return [
        (session_id, session_dates[session_id], items)
        for session_id, items in session_items.items()
    ]


def load_diginetica_sessions(path):
    df = pd.read_csv(
        path,
        sep=";",
        usecols=["sessionId", "itemId", "timeframe", "eventdate"],
    )
    df = df.rename(columns={"sessionId": "session_id", "itemId": "item_id"})
    df["eventdate"] = pd.to_datetime(df["eventdate"], format="%Y-%m-%d")

    session_clicks = {}
    session_dates = {}
    current_session_id = None
    current_date = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_date is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_date

        current_session_id = session_id
        current_date = row.eventdate

        click = (row.item_id, int(row.timeframe))
        if session_id in session_clicks:
            session_clicks[session_id].append(click)
        else:
            session_clicks[session_id] = [click]

    if current_session_id is not None:
        session_dates[current_session_id] = current_date

    sessions = []
    for session_id, clicks in session_clicks.items():
        ordered_clicks = sorted(clicks, key=lambda click: click[1])
        items = [item for item, _ in ordered_clicks]
        sessions.append((session_id, session_dates[session_id], items))
    return sessions

In [ ]:
def drop_short_sessions(sessions):
    return [session for session in sessions if len(session[2]) >= 2]


def drop_rare_items(sessions, min_freq=5):
    counts = Counter()
    for _, _, items in sessions:
        counts.update(items)

    result = []
    for session_id, date, items in sessions:
        kept = [i for i in items if counts[i] >= min_freq]
        if len(kept) >= 2:
            result.append((session_id, date, kept))
    return result


def sort_by_date(sessions):
    return sorted(sessions, key=lambda session: session[1])


def split_by_date(sessions, test_days):
    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)
    train = [s for s in sessions if s[1] < split_date]
    test = [s for s in sessions if s[1] > split_date]
    return train, test


def renumber_training_items(train_sessions):
    item_to_index = {}
    next_item_index = 1
    remapped_sessions = []

    for session_id, date, items in train_sessions:
        remapped_items = []
        for item in items:
            if item not in item_to_index:
                item_to_index[item] = next_item_index
                next_item_index += 1
            remapped_items.append(item_to_index[item])
        remapped_sessions.append((session_id, date, remapped_items))

    return remapped_sessions, item_to_index


def remap_test_sessions(test_sessions, item_to_index):
    remapped_sessions = []
    for session_id, date, items in test_sessions:
        remapped_items = [
            item_to_index[item] for item in items if item in item_to_index
        ]
        if len(remapped_items) >= 2:
            remapped_sessions.append((session_id, date, remapped_items))
    return remapped_sessions


def expand_sessions(sessions):
    examples = []
    for session_id, date, items in sessions:
        for reverse_offset in range(1, len(items)):
            examples.append(
                (session_id, date, items[:-reverse_offset], items[-reverse_offset])
            )
    return examples


def keep_recent_fraction(examples, denominator):
    if denominator is None:
        return examples
    keep = len(examples) // denominator
    return examples[-keep:] if keep else examples


def prefix_label_rows(examples):
    return [(list(prefix), int(label)) for _, _, prefix, label in examples]


def vocabulary_size_from_rows(*row_groups):
    max_item_id = 0
    for rows in row_groups:
        for prefix, label in rows:
            max_item_id = max(max_item_id, int(label), max(prefix))
    return max_item_id + 1


def preprocess(sessions, test_days, train_fraction_denominator):
    sessions = drop_short_sessions(sessions)
    sessions = drop_rare_items(sessions)
    sessions = sort_by_date(sessions)
    train_sessions, test_sessions = split_by_date(sessions, test_days)

    train_sessions, item_to_index = renumber_training_items(train_sessions)
    test_sessions = remap_test_sessions(test_sessions, item_to_index)

    train_examples = expand_sessions(train_sessions)
    test_examples = expand_sessions(test_sessions)
    train_examples = keep_recent_fraction(train_examples, train_fraction_denominator)
    return train_examples, test_examples

In [ ]:
yoochoose_sessions = load_yoochoose_sessions(YOOCHOOSE_SOURCE)
diginetica_sessions = load_diginetica_sessions(DIGINETICA_SOURCE)

yoochoose_1_64_train, yoochoose_1_64_test = preprocess(
    yoochoose_sessions, test_days=1, train_fraction_denominator=64
)
diginetica_train, diginetica_test = preprocess(
    diginetica_sessions, test_days=7, train_fraction_denominator=None
)

YOOCHOOSE_1_64_TRAIN_ROWS = prefix_label_rows(yoochoose_1_64_train)
YOOCHOOSE_1_64_TEST_ROWS = prefix_label_rows(yoochoose_1_64_test)
DIGINETICA_TRAIN_ROWS = prefix_label_rows(diginetica_train)
DIGINETICA_TEST_ROWS = prefix_label_rows(diginetica_test)

print(f"Yoochoose 1/64 train examples: {len(YOOCHOOSE_1_64_TRAIN_ROWS):,}")
print(f"Yoochoose 1/64 test examples: {len(YOOCHOOSE_1_64_TEST_ROWS):,}")
print(f"Diginetica train examples: {len(DIGINETICA_TRAIN_ROWS):,}")
print(f"Diginetica test examples: {len(DIGINETICA_TEST_ROWS):,}")

## Model and Training

To be added.